# fastText dictionary validation: Kiwi count + Kiwi TF-IDF

Objective:

- Read fastText expanded dictionaries from `final/expanded_dictionaries/expanded_dictionary_theta_*.csv`.
- Match the evaluation style of `final/15_kiwi_embedding_seed_expansion.ipynb`.
- Both count and TF-IDF are computed on the Kiwi noun/noun-phrase corpus.
- Compare seed-only and threshold-level fastText expanded dictionaries with the same scoring rules.


In [1]:
from pathlib import Path
import html
import math
import re
import subprocess
import sys
import unicodedata
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from kiwipiepy import Kiwi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy"])
    from kiwipiepy import Kiwi

from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)


## 1. Paths and Settings

In [2]:
if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    Path.cwd(),
    Path.cwd().parent,
]


def first_existing(candidates, label):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"No existing path for {label}: {candidates}")


ROOT = first_existing([p for p in ROOT_CANDIDATES if (p / "data").exists() and (p / "final").exists()], "ROOT")
DATA_DIR = ROOT / "data"
FINAL_DIR = ROOT / "final"
RAW_XML_DIR = first_existing([
    FINAL_DIR / "raw_xml",
    DATA_DIR / "dart" / "raw_xml",
    ROOT / "raw_xml",
], "RAW_XML_DIR")
SEED_DICTIONARY_PATH = first_existing([
    FINAL_DIR / "seed_dictionary.csv",
    DATA_DIR / "seed_dictionary.csv",
    ROOT / "seed_dictionary.csv",
], "SEED_DICTIONARY_PATH")
COMPANY_MASTER_PATH = first_existing([
    FINAL_DIR / "company_master.csv",
    DATA_DIR / "company_master.csv",
    ROOT / "company_master.csv",
], "COMPANY_MASTER_PATH")
EXPANDED_DICT_DIR = FINAL_DIR / "expanded_dictionaries"
OUTPUT_DIR = FINAL_DIR / "fasttext_regex_count_kiwi_tfidf_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# None?? expanded_dictionary_theta_*.csv ??? ????.
REQUESTED_THRESHOLDS = None
MAX_FILES = None

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}
GRADE_BY_DIMENSION = {"E": "e_grade_num", "S": "s_grade_num", "G": "g_grade_num"}

print("ROOT:", ROOT)
print("RAW_XML_DIR:", RAW_XML_DIR)
print("SEED_DICTIONARY_PATH:", SEED_DICTIONARY_PATH)
print("COMPANY_MASTER_PATH:", COMPANY_MASTER_PATH)
print("EXPANDED_DICT_DIR:", EXPANDED_DICT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/UD_26
RAW_XML_DIR: /content/drive/MyDrive/UD_26/final/raw_xml
SEED_DICTIONARY_PATH: /content/drive/MyDrive/UD_26/final/seed_dictionary.csv
COMPANY_MASTER_PATH: /content/drive/MyDrive/UD_26/final/company_master.csv
EXPANDED_DICT_DIR: /content/drive/MyDrive/UD_26/final/expanded_dictionaries
OUTPUT_DIR: /content/drive/MyDrive/UD_26/final/fasttext_regex_count_kiwi_tfidf_validation


## 2. Build DART II/IV/VI Firm-Year Corpus

In [3]:
TARGET_TITLE_REGEX = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}

TITLE_RE = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
MAIN_TITLE_RE = re.compile(
    r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\."
)


def clean_xml_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_text(value) -> str:
    text = "" if pd.isna(value) else str(value)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_term(value) -> str:
    text = normalize_text(value)
    return text.strip(" \t\r\n\"'`.,;:()[]{}<>")


def parse_xml_filename(path: Path) -> tuple[str, int, str]:
    match = re.match(r"(\d{6})_(\d{4})_(\d+)\.xml$", path.name)
    if not match:
        raise ValueError(f"Unexpected XML filename: {path.name}")
    stock_code, fiscal_year, rcept_no = match.groups()
    return stock_code, int(fiscal_year), rcept_no


def extract_target_sections(xml_text: str) -> list[dict]:
    titles = []
    for match in TITLE_RE.finditer(xml_text):
        title = clean_xml_text(match.group(1))
        if MAIN_TITLE_RE.match(title):
            titles.append((title, match.start()))

    sections = []
    for i, (title, start) in enumerate(titles):
        section_name = None
        for name, pattern in TARGET_TITLE_REGEX.items():
            if re.search(pattern, title):
                section_name = name
                break
        if section_name is None:
            continue
        end = titles[i + 1][1] if i + 1 < len(titles) else len(xml_text)
        sections.append({"section": section_name, "text": clean_xml_text(xml_text[start:end])})
    return sections


def load_company_names(path: Path) -> pd.DataFrame:
    company_df = pd.read_csv(path, dtype={"stock_code": "string"}, encoding="utf-8-sig")
    company_df["stock_code"] = company_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
    if "company_name" not in company_df.columns:
        return pd.DataFrame(columns=["stock_code", "company_name"])
    return company_df[["stock_code", "company_name"]].dropna().drop_duplicates("stock_code")


def build_firm_year_corpus(raw_xml_dir: Path, max_files: int | None = None) -> pd.DataFrame:
    rows = []
    xml_files = sorted(raw_xml_dir.glob("*.xml"))
    if max_files is not None:
        xml_files = xml_files[:max_files]
    if not xml_files:
        raise FileNotFoundError(f"No XML files found in {raw_xml_dir}")

    for xml_path in xml_files:
        stock_code, fiscal_year, rcept_no = parse_xml_filename(xml_path)
        xml_text = xml_path.read_text(encoding="utf-8", errors="ignore")
        sections = extract_target_sections(xml_text)
        document = " ".join(section["text"] for section in sections)
        document_norm = normalize_text(document)
        rows.append({
            "stock_code": stock_code,
            "fiscal_year": fiscal_year,
            "rcept_no": rcept_no,
            "file_name": xml_path.name,
            "document": document,
            "document_norm": document_norm,
            "section_count": len({section["section"] for section in sections}),
            "total_word_count": len(document_norm.split()),
            "total_char_count": len(document_norm),
            "esg_year": fiscal_year + 1,
        })

    corpus = pd.DataFrame(rows)
    corpus = corpus.merge(load_company_names(COMPANY_MASTER_PATH), on="stock_code", how="left")
    cols = ["stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no", "file_name", "document", "document_norm", "section_count", "total_word_count", "total_char_count"]
    return corpus[cols].sort_values(["stock_code", "fiscal_year", "rcept_no"]).reset_index(drop=True)


corpus_df = build_firm_year_corpus(RAW_XML_DIR, MAX_FILES)
print("corpus rows:", len(corpus_df))
print(corpus_df["section_count"].value_counts().sort_index())
display(corpus_df[["stock_code", "company_name", "fiscal_year", "esg_year", "section_count", "total_word_count"]].head())


corpus rows: 381
section_count
3    381
Name: count, dtype: int64


,stock_code,company_name,fiscal_year,esg_year,section_count,total_word_count
0,000020,동화약품,2022,2023,3,7863
1,000020,동화약품,2023,2024,3,8120
2,000020,동화약품,2024,2025,3,8152
3,000040,KR모터스,2022,2023,3,4256
4,000040,KR모터스,2023,2024,3,4943


## 3. Load Seed and fastText Expanded Dictionaries

In [4]:
def threshold_label(theta: float) -> str:
    return f"{float(theta):.2f}".replace(".", "_")


def pattern_from_term(term: str) -> str:
    term = normalize_term(term)
    if not term:
        return r"a^"
    escaped = re.escape(term)
    escaped = escaped.replace(r"\ ", r"\s+")
    return escaped


def split_seed_terms(row: pd.Series) -> list[str]:
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))
    terms = []
    seen = set()
    for value in values:
        term = normalize_term(value)
        if not term or term.lower() == "nan" or term in seen:
            continue
        seen.add(term)
        terms.append(term)
    return terms


def build_seed_dictionary(seed_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for idx, row in seed_df.reset_index(drop=True).iterrows():
        dimension = normalize_term(row.get("dimension", ""))
        if dimension not in {"E", "S", "G"}:
            continue
        seed_term = normalize_term(row.get("seed_term", ""))
        for term in split_seed_terms(row):
            rows.append({
                "dictionary": "kiwi_seed_only",
                "threshold": np.nan,
                "dimension": dimension,
                "source": "seed",
                "seed_term": seed_term,
                "candidate_term": term,
                "pattern": pattern_from_term(term),
                "similarity": 1.0,
            })
    return pd.DataFrame(rows).drop_duplicates(["dimension", "candidate_term"]).reset_index(drop=True)


def parse_threshold_from_path(path: Path) -> float:
    match = re.search(r"theta_(\d+)_(\d+)", path.stem)
    if not match:
        raise ValueError(f"Cannot parse threshold from {path.name}")
    return float(f"{match.group(1)}.{match.group(2)}")


seed_source_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
seed_dictionary_df = build_seed_dictionary(seed_source_df)

expanded_paths = sorted(EXPANDED_DICT_DIR.glob("expanded_dictionary_theta_*.csv"))
if not expanded_paths:
    raise FileNotFoundError(
        f"No expanded_dictionary_theta_*.csv files found in {EXPANDED_DICT_DIR}. "
        "Run final/03 or final/04 fastText dictionary generation first."
    )

if REQUESTED_THRESHOLDS is not None:
    requested = {round(float(theta), 2) for theta in REQUESTED_THRESHOLDS}
    expanded_paths = [path for path in expanded_paths if round(parse_threshold_from_path(path), 2) in requested]

expanded_dicts = {}
for path in expanded_paths:
    theta = parse_threshold_from_path(path)
    df = pd.read_csv(path, encoding="utf-8-sig")
    required = {"dimension", "candidate_term"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing columns: {sorted(missing)}")
    usable = df.copy()
    if "keep_review" in usable.columns:
        usable = usable[usable["keep_review"].astype(str).str.upper() != "FALSE"]
    usable = usable[usable["dimension"].isin(["E", "S", "G"])]
    usable["candidate_term"] = usable["candidate_term"].map(normalize_term)
    usable = usable[usable["candidate_term"].str.len() > 0]
    if "pattern" not in usable.columns:
        usable["pattern"] = usable["candidate_term"].map(pattern_from_term)
    usable["pattern"] = usable.apply(
        lambda row: pattern_from_term(row["candidate_term"]) if pd.isna(row.get("pattern")) or not str(row.get("pattern")).strip() else str(row.get("pattern")),
        axis=1,
    )
    usable["dictionary"] = f"fasttext_theta_{threshold_label(theta)}"
    usable["threshold"] = theta
    if "source" not in usable.columns:
        usable["source"] = "fasttext_candidate"
    if "seed_term" not in usable.columns:
        usable["seed_term"] = ""
    if "similarity" not in usable.columns:
        usable["similarity"] = np.nan
    cols = ["dictionary", "threshold", "dimension", "source", "seed_term", "candidate_term", "pattern", "similarity"]
    expanded_dicts[theta] = usable[cols].drop_duplicates(["dimension", "candidate_term"]).reset_index(drop=True)

print("seed terms:")
display(seed_dictionary_df.groupby("dimension").size().rename("term_count").reset_index())
print("expanded dictionary files:", len(expanded_dicts))
display(pd.DataFrame([
    {"threshold": theta, "dictionary": df["dictionary"].iloc[0], "terms": len(df), **df.groupby("dimension").size().to_dict()}
    for theta, df in expanded_dicts.items()
]).sort_values("threshold"))


seed terms:


,dimension,term_count
0,E,18
1,G,17
2,S,18


expanded dictionary files: 30


,threshold,dictionary,terms,E,G,S
0,0.10,fasttext_theta_0_10,21259,6056,7269,7934
1,0.20,fasttext_theta_0_20,21259,6056,7269,7934
2,0.30,fasttext_theta_0_30,21259,6056,7269,7934
3,0.40,fasttext_theta_0_40,16800,4867,5537,6396
4,0.45,fasttext_theta_0_45,8496,3074,2038,3384
5,0.50,fasttext_theta_0_50,4465,1630,1125,1710
6,0.55,fasttext_theta_0_55,2469,954,685,830
7,0.56,fasttext_theta_0_56,2114,916,543,655
8,0.57,fasttext_theta_0_57,1793,875,423,495
9,0.58,fasttext_theta_0_58,1524,848,294,382


## 4. Kiwi Count and TF-IDF Feature Generation

In [5]:
# Count and TF-IDF features are generated after the Kiwi corpus is built in the next section.
# This cell is intentionally left as an execution marker so the notebook flow stays explicit.


## 5. Build Kiwi Noun/Noun-Phrase Corpus

In [6]:
kiwi = Kiwi()
NOUN_TAGS = {"NNG", "NNP", "SL"}


def kiwi_noun_candidates(text: str) -> list[str]:
    tokens = kiwi.tokenize(text)
    terms = []
    current = []
    for token in tokens:
        form = normalize_term(token.form)
        tag = token.tag
        if tag in NOUN_TAGS and len(form) > 1:
            terms.append(form)
            current.append(form)
        else:
            if len(current) >= 2:
                phrase = normalize_term(" ".join(current))
                if phrase:
                    terms.append(phrase)
            current = []
    if len(current) >= 2:
        phrase = normalize_term(" ".join(current))
        if phrase:
            terms.append(phrase)
    return terms


def token_key(term: str) -> str:
    return normalize_term(term).replace(" ", "_")


kiwi_terms_by_doc = [kiwi_noun_candidates(text) for text in corpus_df["document_norm"]]
kiwi_doc_df = corpus_df[["stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no", "total_word_count"]].copy()
kiwi_doc_df["kiwi_terms"] = kiwi_terms_by_doc
kiwi_doc_df["kiwi_document"] = [" ".join(token_key(term) for term in terms) for terms in kiwi_terms_by_doc]
kiwi_doc_df["kiwi_term_count"] = [len(terms) for terms in kiwi_terms_by_doc]

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=1,
    norm=None,
)
tfidf_matrix = vectorizer.fit_transform(kiwi_doc_df["kiwi_document"])
vocab = vectorizer.vocabulary_
print("tfidf_matrix:", tfidf_matrix.shape)
print("vocabulary size:", len(vocab))
display(kiwi_doc_df[["stock_code", "company_name", "fiscal_year", "kiwi_term_count", "kiwi_document"]].head())


tfidf_matrix: (381, 249487)
vocabulary size: 249487


,stock_code,company_name,fiscal_year,kiwi_term_count,kiwi_document
0,000020,동화약품,2022,8222,II 사업 II_사업 내용 사업 개요 일반 사항 지배 기업 사항_지배_기업 연결 실체 연결_실체 제공 재화 용역 근거 영업 부문 영업_부문 구분 부문 재무 정보 재무_정보 내부 관리 목적 내부_관리_목적 활용 당사 제약 의료 기기 제조 의료_기기_제조 판매업 기타 금융업 기타_금융업 운영 영위 영업 부문 영업_부문...
1,000020,동화약품,2023,8251,II 사업 II_사업 내용 사업 개요 일반 사항 지배 기업 사항_지배_기업 연결 실체 연결_실체 제공 재화 용역 근거 영업 부문 영업_부문 구분 부문 재무 정보 재무_정보 내부 관리 목적 내부_관리_목적 활용 당사 제약 의료 기기 제조 의료_기기_제조 판매업 기타 금융업 기타_금융업 운영 영위 영업 부문 영업_부문...
2,000020,동화약품,2024,8429,II 사업 II_사업 내용 사업 개요 일반 사항 지배 기업 사항_지배_기업 연결 실체 연결_실체 제공 재화 용역 근거 영업 부문 영업_부문 구분 부문 재무 정보 재무_정보 내부 관리 목적 내부_관리_목적 활용 당사 제약 의료 기기 제조 의료_기기_제조 판매업 기타 금융업 기타_금융업 운영 영위 영업 부문 영업_부문...
3,000040,KR모터스,2022,4507,II 사업 II_사업 내용 사업 개요 업계 현황 수출 주력 시장 현황_수출_주력_시장 유럽 경기 배기량 침체 지속 가운데 배기량 스쿠터 선호도 배기량_스쿠터_선호도 아시아 인도 중국 동남아시아 중심 시장 성장세 지속 전망 인도 시장 인도_시장 세계 수요 세계_수요 차지 최대 시장 최대_시장 부상 배기량 위주 배기량...
4,000040,KR모터스,2023,4945,II 사업 II_사업 내용 사업 개요 업계 현황 수출 주력 시장 현황_수출_주력_시장 유럽 경기 배기량 침체 지속 가운데 배기량 스쿠터 선호도 배기량_스쿠터_선호도 아시아 인도 중국 동남아시아 중심 시장 성장세 지속 전망 인도 시장 인도_시장 세계 수요 세계_수요 차지 최대 시장 최대_시장 부상 배기량 위주 배기량...


In [7]:
def terms_present_in_vocab(terms: list[str]) -> list[str]:
    present = []
    used = set()
    for term in terms:
        term = normalize_term(term)
        key = token_key(term)
        if term and key in vocab and term not in used:
            present.append(term)
            used.add(term)
    return present


def tfidf_sum_for_terms(terms: list[str]) -> np.ndarray:
    terms = terms_present_in_vocab(terms)
    if not terms:
        return np.zeros(tfidf_matrix.shape[0], dtype=float)
    cols = [vocab[token_key(term)] for term in terms]
    return np.asarray(tfidf_matrix[:, cols].sum(axis=1)).ravel()


def count_for_terms(terms: list[str]) -> np.ndarray:
    terms = terms_present_in_vocab(terms)
    if not terms:
        return np.zeros(len(kiwi_doc_df), dtype=int)
    term_set = set(terms)
    return np.asarray([sum(1 for term in doc_terms if term in term_set) for doc_terms in kiwi_doc_df["kiwi_terms"]], dtype=int)


def add_dictionary_features(score_df: pd.DataFrame, feature_meta_rows: list[dict],
                            dictionary: str, threshold, dimension: str,
                            terms: list[str], prefix: str) -> None:
    terms = [normalize_term(term) for term in terms]
    terms = [term for term in terms if term]
    present_terms = terms_present_in_vocab(terms)
    score_df[f"{dimension}_{prefix}_tfidf"] = tfidf_sum_for_terms(terms)
    score_df[f"{dimension}_{prefix}_count"] = count_for_terms(terms)
    feature_meta_rows.append({
        "dictionary": dictionary,
        "threshold": threshold,
        "dimension": dimension,
        "prefix": prefix,
        "term_count": len(present_terms),
        "dictionary_term_count": len(set(terms)),
        "terms_in_vocab": "|".join(present_terms),
    })


score_df = kiwi_doc_df[["stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no", "total_word_count", "kiwi_term_count"]].copy()
feature_meta_rows = []

# Seed-only features.
for dimension in ["E", "S", "G"]:
    terms = seed_dictionary_df.loc[seed_dictionary_df["dimension"].eq(dimension), "candidate_term"].tolist()
    add_dictionary_features(score_df, feature_meta_rows, "seed", np.nan, dimension, terms, "seed")

# Threshold-level fastText expanded features.
for theta, dictionary_df in sorted(expanded_dicts.items()):
    theta_label = threshold_label(theta)
    prefix = f"fasttext_{theta_label}"
    for dimension in ["E", "S", "G"]:
        terms = dictionary_df.loc[dictionary_df["dimension"].eq(dimension), "candidate_term"].tolist()
        add_dictionary_features(score_df, feature_meta_rows, f"fasttext_{theta_label}", theta, dimension, terms, prefix)

# Aggregate ESG features.
for prefix in ["seed"] + [f"fasttext_{threshold_label(theta)}" for theta in sorted(expanded_dicts)]:
    for suffix in ["tfidf", "count"]:
        score_df[f"ESG_{prefix}_{suffix}"] = sum(score_df[f"{dimension}_{prefix}_{suffix}"] for dimension in ["E", "S", "G"])

feature_meta_df = pd.DataFrame(feature_meta_rows)
score_path = OUTPUT_DIR / "fasttext_kiwi_threshold_scores.csv"
meta_path = OUTPUT_DIR / "fasttext_kiwi_threshold_feature_terms.csv"
score_df.to_csv(score_path, index=False, encoding="utf-8-sig")
feature_meta_df.to_csv(meta_path, index=False, encoding="utf-8-sig")
print("saved:", score_path)
print("saved:", meta_path)
display(feature_meta_df[["dictionary", "threshold", "dimension", "term_count", "dictionary_term_count", "prefix"]])


saved: /content/drive/MyDrive/UD_26/final/fasttext_regex_count_kiwi_tfidf_validation/fasttext_kiwi_threshold_scores.csv
saved: /content/drive/MyDrive/UD_26/final/fasttext_regex_count_kiwi_tfidf_validation/fasttext_kiwi_threshold_feature_terms.csv


,dictionary,threshold,dimension,term_count,dictionary_term_count,prefix
0,seed,NaN,E,10,18,seed
1,seed,NaN,S,12,18,seed
2,seed,NaN,G,9,17,seed
3,fasttext_0_10,0.1,E,563,6056,fasttext_0_10
4,fasttext_0_10,0.1,S,642,7934,fasttext_0_10
...,...,...,...,...,...,...
88,fasttext_0_90,0.9,S,12,18,fasttext_0_90
89,fasttext_0_90,0.9,G,9,17,fasttext_0_90
90,fasttext_1_00,1.0,E,10,18,fasttext_1_00
91,fasttext_1_00,1.0,S,12,18,fasttext_1_00


## 6. Merge ESG Grades

In [8]:
def normalize_stock_code(series: pd.Series) -> pd.Series:
    return series.astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)


analysis_df = score_df.copy()
company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
company_master["stock_code"] = normalize_stock_code(company_master["stock_code"])
analysis_df["stock_code"] = normalize_stock_code(analysis_df["stock_code"])
for col in ["fiscal_year", "esg_year"]:
    company_master[col] = pd.to_numeric(company_master[col], errors="coerce").astype("Int64")
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce").astype("Int64")

for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    company_master[f"{col}_num"] = company_master[col].map(GRADE_MAP)

grade_cols = [
    "stock_code", "fiscal_year", "esg_year", "industry",
    "esg_grade", "e_grade", "s_grade", "g_grade",
    "esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num",
]
grade_df = company_master[grade_cols].drop_duplicates(["stock_code", "fiscal_year", "esg_year"])
analysis_df = analysis_df.merge(grade_df, on=["stock_code", "fiscal_year", "esg_year"], how="left")
analysis_path = OUTPUT_DIR / "fasttext_kiwi_validation_analysis_panel.csv"
analysis_df.to_csv(analysis_path, index=False, encoding="utf-8-sig")
print("analysis rows:", len(analysis_df))
print("missing esg_grade_num:", analysis_df["esg_grade_num"].isna().sum())
print("saved:", analysis_path)
display(analysis_df[["stock_code", "company_name", "fiscal_year", "esg_grade", "e_grade", "s_grade", "g_grade"]].head())


analysis rows: 381
missing esg_grade_num: 0
saved: /content/drive/MyDrive/UD_26/final/fasttext_regex_count_kiwi_tfidf_validation/fasttext_kiwi_validation_analysis_panel.csv


,stock_code,company_name,fiscal_year,esg_grade,e_grade,s_grade,g_grade
0,000020,동화약품,2022,C,C,B,C
1,000020,동화약품,2023,C,B,B,C
2,000020,동화약품,2024,C,B,C,C
3,000040,KR모터스,2022,D,D,D,D
4,000040,KR모터스,2023,D,D,D,D


## 7. Pooled Spearman Correlations

In [9]:
def safe_spearman(x: pd.Series, y: pd.Series) -> tuple[float, float, int]:
    data = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 3 or data["x"].nunique() < 2 or data["y"].nunique() < 2:
        return np.nan, np.nan, len(data)
    rho, pvalue = spearmanr(data["x"], data["y"])
    return float(rho), float(pvalue), len(data)


def add_corr(rows, dictionary, threshold, dimension, score_type, feature, grade_col, term_count):
    rho, pvalue, n = safe_spearman(analysis_df[feature], analysis_df[grade_col])
    rows.append({
        "dictionary": dictionary,
        "threshold": threshold,
        "dimension": dimension,
        "score_type": score_type,
        "feature": feature,
        "grade_col": grade_col,
        "term_count": term_count,
        "spearman_rho": rho,
        "p_value": pvalue,
        "n": n,
    })


corr_rows = []
for _, meta in feature_meta_df.iterrows():
    dimension = meta["dimension"]
    grade_col = GRADE_BY_DIMENSION[dimension]
    prefix = meta["prefix"]
    for score_type, suffix in [("tfidf", "tfidf"), ("count", "count")]:
        feature = f"{dimension}_{prefix}_{suffix}"
        add_corr(corr_rows, meta["dictionary"], meta["threshold"], dimension, score_type, feature, grade_col, int(meta["term_count"]))

for dictionary in ["seed"] + [f"fasttext_{threshold_label(theta)}" for theta in sorted(expanded_dicts)]:
    if dictionary == "seed":
        theta = np.nan
        prefix = "seed"
    else:
        theta = float(dictionary.split("_")[-2] + "." + dictionary.split("_")[-1])
        prefix = dictionary
    meta_sub = feature_meta_df[feature_meta_df["prefix"].eq(prefix)]
    term_count = int(meta_sub["term_count"].sum())
    for score_type, suffix in [("tfidf", "tfidf"), ("count", "count")]:
        feature = f"ESG_{prefix}_{suffix}"
        add_corr(corr_rows, dictionary, theta, "ESG", score_type, feature, "esg_grade_num", term_count)

correlation_df = pd.DataFrame(corr_rows).sort_values(
    ["dimension", "score_type", "dictionary", "threshold"],
    na_position="first",
)
corr_path = OUTPUT_DIR / "fasttext_kiwi_threshold_spearman.csv"
correlation_df.to_csv(corr_path, index=False, encoding="utf-8-sig")
print("saved:", corr_path)
display(correlation_df)


saved: /content/drive/MyDrive/UD_26/final/fasttext_regex_count_kiwi_tfidf_validation/fasttext_kiwi_threshold_spearman.csv


,dictionary,threshold,dimension,score_type,feature,grade_col,term_count,spearman_rho,p_value,n
7,fasttext_0_10,0.10,E,count,E_fasttext_0_10_count,e_grade_num,563,0.500714,1.428989e-25,381
13,fasttext_0_20,0.20,E,count,E_fasttext_0_20_count,e_grade_num,563,0.500714,1.428989e-25,381
19,fasttext_0_30,0.30,E,count,E_fasttext_0_30_count,e_grade_num,563,0.500714,1.428989e-25,381
25,fasttext_0_40,0.40,E,count,E_fasttext_0_40_count,e_grade_num,405,0.492172,1.226458e-24,381
31,fasttext_0_45,0.45,E,count,E_fasttext_0_45_count,e_grade_num,215,0.560730,6.228455e-33,381
...,...,...,...,...,...,...,...,...,...,...
164,fasttext_0_75,0.75,S,tfidf,S_fasttext_0_75_tfidf,s_grade_num,12,0.201298,7.594277e-05,381
170,fasttext_0_80,0.80,S,tfidf,S_fasttext_0_80_tfidf,s_grade_num,12,0.201298,7.594277e-05,381
176,fasttext_0_90,0.90,S,tfidf,S_fasttext_0_90_tfidf,s_grade_num,12,0.201298,7.594277e-05,381
182,fasttext_1_00,1.00,S,tfidf,S_fasttext_1_00_tfidf,s_grade_num,12,0.201298,7.594277e-05,381


In [10]:
for score_type in ["tfidf", "count"]:
    pivot = correlation_df[
        ~correlation_df["dictionary"].eq("seed")
        & correlation_df["score_type"].eq(score_type)
    ].pivot_table(
        index="threshold",
        columns="dimension",
        values="spearman_rho",
        aggfunc="first",
    )
    print(f"Expanded fastText Spearman rho by threshold - {score_type}")
    display(pivot)

print("Seed-only baseline")
display(correlation_df[correlation_df["dictionary"].eq("seed")])

print("Expanded dictionary term counts observed in Kiwi vocabulary")
display(
    feature_meta_df[~feature_meta_df["dictionary"].eq("seed")]
    .pivot(index="threshold", columns="dimension", values="term_count")
)


Expanded fastText Spearman rho by threshold - tfidf


dimension,E,ESG,G,S
threshold,,,,
0.10,0.126381,0.450695,0.303398,0.375110
0.20,0.126381,0.450695,0.303398,0.375110
0.30,0.126381,0.450695,0.303398,0.375110
0.40,0.141603,0.427363,0.263890,0.334419
0.45,0.234652,0.415964,0.227669,0.316706
0.50,0.371998,0.320834,-0.048757,0.177804
0.55,0.409166,0.187833,-0.157805,0.283135
0.56,0.409105,0.187680,-0.157616,0.283135
0.57,0.406977,0.182410,-0.158851,0.269505


Expanded fastText Spearman rho by threshold - count


dimension,E,ESG,G,S
threshold,,,,
0.10,0.500714,0.693858,0.655878,0.626059
0.20,0.500714,0.693858,0.655878,0.626059
0.30,0.500714,0.693858,0.655878,0.626059
0.40,0.492172,0.708034,0.668882,0.625296
0.45,0.560730,0.711037,0.658557,0.614237
0.50,0.590051,0.725817,0.648029,0.550504
0.55,0.513502,0.693299,0.578611,0.652923
0.56,0.513565,0.693349,0.578641,0.652923
0.57,0.512530,0.693247,0.578726,0.652717


Seed-only baseline


,dictionary,threshold,dimension,score_type,feature,grade_col,term_count,spearman_rho,p_value,n
1,seed,NaN,E,count,E_seed_count,e_grade_num,10,0.511493,8.685163e-27,381
0,seed,NaN,E,tfidf,E_seed_tfidf,e_grade_num,10,0.410174,6.811972e-17,381
187,seed,NaN,ESG,count,ESG_seed_count,esg_grade_num,31,0.691826,1.483717e-55,381
186,seed,NaN,ESG,tfidf,ESG_seed_tfidf,esg_grade_num,31,0.117501,2.179423e-02,381
5,seed,NaN,G,count,G_seed_count,g_grade_num,9,0.568412,5.504282e-34,381
4,seed,NaN,G,tfidf,G_seed_tfidf,g_grade_num,9,-0.185968,2.623377e-04,381
3,seed,NaN,S,count,S_seed_count,s_grade_num,12,0.638957,4.200036e-45,381
2,seed,NaN,S,tfidf,S_seed_tfidf,s_grade_num,12,0.201298,7.594277e-05,381


Expanded dictionary term counts observed in Kiwi vocabulary


dimension,E,G,S
threshold,,,
0.10,563,497,642
0.20,563,497,642
0.30,563,497,642
0.40,405,358,441
0.45,215,129,190
0.50,85,57,71
0.55,24,24,25
0.56,22,23,25
0.57,19,20,23


## 8. Year-by-Year Spearman Robustness

In [11]:
year_rows = []
for _, row in correlation_df.iterrows():
    feature = row["feature"]
    grade_col = row["grade_col"]
    if feature not in analysis_df.columns or grade_col not in analysis_df.columns:
        continue
    for fiscal_year, sub in analysis_df.groupby("fiscal_year"):
        rho, pvalue, n = safe_spearman(sub[feature], sub[grade_col])
        year_rows.append({
            "dictionary": row["dictionary"],
            "threshold": row["threshold"],
            "dimension": row["dimension"],
            "score_type": row["score_type"],
            "fiscal_year": fiscal_year,
            "feature": feature,
            "grade_col": grade_col,
            "term_count": row["term_count"],
            "spearman_rho": rho,
            "p_value": pvalue,
            "n": n,
        })

year_correlation_df = pd.DataFrame(year_rows).sort_values(
    ["dimension", "score_type", "dictionary", "threshold", "fiscal_year"],
    na_position="first",
)
year_corr_path = OUTPUT_DIR / "fasttext_kiwi_threshold_spearman_by_year.csv"
year_correlation_df.to_csv(year_corr_path, index=False, encoding="utf-8-sig")
print("saved:", year_corr_path)
display(year_correlation_df)

for score_type in ["tfidf", "count"]:
    year_pivot = year_correlation_df[
        ~year_correlation_df["dictionary"].eq("seed")
        & year_correlation_df["score_type"].eq(score_type)
    ].pivot_table(
        index=["threshold", "fiscal_year"],
        columns="dimension",
        values="spearman_rho",
        aggfunc="first",
    )
    print(f"Year-by-year expanded fastText Spearman rho - {score_type}")
    display(year_pivot)


saved: /content/drive/MyDrive/UD_26/final/fasttext_regex_count_kiwi_tfidf_validation/fasttext_kiwi_threshold_spearman_by_year.csv


,dictionary,threshold,dimension,score_type,fiscal_year,feature,grade_col,term_count,spearman_rho,p_value,n
0,fasttext_0_10,0.1,E,count,2022,E_fasttext_0_10_count,e_grade_num,563,0.541018,5.148919e-11,127
1,fasttext_0_10,0.1,E,count,2023,E_fasttext_0_10_count,e_grade_num,563,0.527612,1.846427e-10,127
2,fasttext_0_10,0.1,E,count,2024,E_fasttext_0_10_count,e_grade_num,563,0.424946,6.369951e-07,127
3,fasttext_0_20,0.2,E,count,2022,E_fasttext_0_20_count,e_grade_num,563,0.541018,5.148919e-11,127
4,fasttext_0_20,0.2,E,count,2023,E_fasttext_0_20_count,e_grade_num,563,0.527612,1.846427e-10,127
...,...,...,...,...,...,...,...,...,...,...,...
739,fasttext_1_00,1.0,S,tfidf,2023,S_fasttext_1_00_tfidf,s_grade_num,12,0.178507,4.465035e-02,127
740,fasttext_1_00,1.0,S,tfidf,2024,S_fasttext_1_00_tfidf,s_grade_num,12,0.196162,2.708432e-02,127
741,seed,NaN,S,tfidf,2022,S_seed_tfidf,s_grade_num,12,0.228137,9.889631e-03,127
742,seed,NaN,S,tfidf,2023,S_seed_tfidf,s_grade_num,12,0.178507,4.465035e-02,127


Year-by-year expanded fastText Spearman rho - tfidf


dimension                     E       ESG         G         S
threshold fiscal_year                                        
0.1       2022         0.162485  0.417709  0.323739  0.388486
          2023         0.151069  0.479417  0.295346  0.367435
          2024         0.064662  0.450849  0.298136  0.358738
0.2       2022         0.162485  0.417709  0.323739  0.388486
          2023         0.151069  0.479417  0.295346  0.367435
...                         ...       ...       ...       ...
0.9       2023         0.426297  0.097126 -0.226198  0.178507
          2024         0.353616  0.121741 -0.172107  0.196162
1.0       2022         0.440858  0.129022 -0.164460  0.228137
          2023         0.426297  0.097126 -0.226198  0.178507
          2024         0.353616  0.121741 -0.172107  0.196162

[90 rows x 4 columns]

Year-by-year expanded fastText Spearman rho - count


dimension                     E       ESG         G         S
threshold fiscal_year                                        
0.1       2022         0.541018  0.710666  0.677159  0.690712
          2023         0.527612  0.710873  0.672312  0.620520
          2024         0.424946  0.653825  0.620879  0.559277
0.2       2022         0.541018  0.710666  0.677159  0.690712
          2023         0.527612  0.710873  0.672312  0.620520
...                         ...       ...       ...       ...
0.9       2023         0.535100  0.702254  0.570807  0.620840
          2024         0.452377  0.666503  0.550289  0.568705
1.0       2022         0.542982  0.708334  0.601902  0.715241
          2023         0.535100  0.702254  0.570807  0.620840
          2024         0.452377  0.666503  0.550289  0.568705

[90 rows x 4 columns]